# NC-SRAG P3: LLM-judge validation of naturalistic labels

**Run this AFTER both P2 runs finish** (Qwen-7B and Mistral-7B), so it judges all
four `results_squad_*.json` files at once. A strong, larger independent model
(Qwen2.5-14B-Instruct by default) re-labels every naturalistic generation as
correct / hallucination / abstain. We then report agreement and Cohen's kappa
against the exact-match labels, export every disagreement for human spot-check,
and build a stratified human-annotation sheet for the pre-registered kappa>=0.7 gate.

**Runtime:** Colab A100 (14B in bf16 needs ~28 GB). Run all.

In [ ]:
# 1. install + mount
!pip -q install transformers accelerate datasets scikit-learn scipy bitsandbytes
from google.colab import drive; drive.mount('/content/drive')
import os
# os.environ['JUDGE_MODEL']='Qwen/Qwen2.5-14B-Instruct'  # swap judge here if desired

In [ ]:
%%writefile judge.py
import os, re, json, glob, time
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
WORK="/content/drive/MyDrive/ncsrag_nat"
JUDGE=os.environ.get("JUDGE_MODEL","Qwen/Qwen2.5-14B-Instruct")
print("judge model:",JUDGE,flush=True)
tok=AutoTokenizer.from_pretrained(JUDGE)
from transformers import BitsAndBytesConfig
_bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(JUDGE,quantization_config=_bnb,device_map="auto").eval()

SYS=("You are a strict grader for short-answer question answering. You are given a "
 "question, a reference answer that is known to be correct, and a candidate answer "
 "produced by another system. Decide whether the candidate is correct. The candidate "
 "is CORRECT if it conveys the same factual answer as the reference, ignoring phrasing, "
 "extra words, capitalisation, or added explanation. It is WRONG if it gives a different "
 "or contradictory answer. It is REFUSED if it declines, says it does not know, or says "
 "the passages do not contain the answer. Reply with exactly one word: CORRECT, WRONG, or REFUSED.")

def make_prompt(q,gold,cand):
    user=f"Question: {q}\nReference answer: {gold}\nCandidate answer: {cand}\nYour one-word verdict:"
    try:
        return tok.apply_chat_template([{"role":"system","content":SYS},{"role":"user","content":user}],
                                       tokenize=False,add_generation_prompt=True)
    except Exception:
        return tok.apply_chat_template([{"role":"user","content":SYS+"\n\n"+user}],
                                       tokenize=False,add_generation_prompt=True)

@torch.no_grad()
def verdict(q,gold,cand):
    enc=tok(make_prompt(q,gold,cand),return_tensors="pt",truncation=True,max_length=2048).to(model.device)
    out=model.generate(**enc,max_new_tokens=4,do_sample=False,pad_token_id=tok.eos_token_id)
    t=tok.decode(out[0][enc["input_ids"].shape[1]:],skip_special_tokens=True).upper()
    if "REFUS" in t or "UNKNOWN" in t: return "abstain"
    if "INCORRECT" in t or "WRONG" in t: return "hallucination"
    if "CORRECT" in t: return "correct"
    return "hallucination"

def kappa(a,b,labels):
    idx={l:i for i,l in enumerate(labels)}; n=len(a); k=len(labels)
    if n==0: return float("nan"), np.zeros((k,k))
    m=np.zeros((k,k))
    for x,y in zip(a,b): m[idx[x],idx[y]]+=1
    po=np.trace(m)/n
    pe=sum((m[i,:].sum()/n)*(m[:,i].sum()/n) for i in range(k))
    return ((po-pe)/(1-pe) if pe<1 else 1.0), m

def run():
    files=sorted(glob.glob(f"{WORK}/results_squad_*.json"))
    files=[f for f in files if "_partial" not in f]
    print("files to judge:",[os.path.basename(f) for f in files],flush=True)
    summary={}
    for f in files:
        rows=json.load(open(f)); tag=os.path.basename(f)[len("results_"):-len(".json")]
        part=f"{WORK}/judged_{tag}_partial.json"
        out=json.load(open(part)) if os.path.exists(part) else []
        done={r["qid"]+"_"+r["cond"] for r in out}; t0=time.time()
        for i,r in enumerate(rows):
            if r["qid"]+"_"+r["cond"] in done: continue
            jl=verdict(r["question"],r["gold"],r["answer"])
            nr=dict(r); nr["label_em"]=r["label"]; nr["label"]=jl; out.append(nr)
            if i%200==0:
                print(f"  {tag}: {i}/{len(rows)} t={time.time()-t0:.0f}s",flush=True)
                json.dump(out,open(part,"w"))
        json.dump(out,open(f"{WORK}/judged_{tag}.json","w"))
        em=[r["label_em"] for r in out]; jd=[r["label"] for r in out]
        L=["correct","hallucination","abstain"]; kp3,cm=kappa(em,jd,L)
        mask=[(e!="abstain" and j!="abstain") for e,j in zip(em,jd)]
        em2=[e for e,m in zip(em,mask) if m]; jd2=[j for j,m in zip(jd,mask) if m]
        kp2,_=kappa(em2,jd2,["correct","hallucination"])
        fH_C=sum(1 for e,j in zip(em,jd) if e=="hallucination" and j=="correct")
        fC_H=sum(1 for e,j in zip(em,jd) if e=="correct" and j=="hallucination")
        agree=sum(1 for e,j in zip(em,jd) if e==j)/max(1,len(em))
        summary[tag]=dict(n=len(out),agreement=agree,kappa3=kp3,kappa2_corr_halluc=kp2,
                          em_halluc_judge_correct=fH_C,em_correct_judge_halluc=fC_H,
                          confusion={f"em_{a}->judge_{b}":int(cm[i,k]) for i,a in enumerate(L) for k,b in enumerate(L)})
        print(f"== {tag}: agree={agree:.3f} kappa3={kp3:.3f} kappa2(c/h)={kp2:.3f} "
              f"EMhalluc->judgeCorrect={fH_C} EMcorrect->judgeHalluc={fC_H}",flush=True)
        dis=[dict(qid=r["qid"],cond=r["cond"],question=r["question"],gold=r["gold"],answer=r["answer"],
                  label_em=r["label_em"],label_judge=r["label"]) for r in out if r["label_em"]!=r["label"]]
        json.dump(dis,open(f"{WORK}/disagreements_{tag}.json","w"))
    json.dump(summary,open(f"{WORK}/judge_summary.json","w"),indent=1)
    print("\nSUMMARY\n"+json.dumps(summary,indent=1),flush=True)
    print("DONE. judged_*.json + disagreements_*.json + judge_summary.json in",WORK,flush=True)


In [ ]:
# 2. judge every naturalistic results file (checkpointed; resumes after disconnect)
import importlib, judge; importlib.reload(judge)
judge.run()

In [ ]:
# build a 120-item stratified human-annotation sheet for the kappa>=0.7 gate
import json, glob, os, random, csv
WORK="/content/drive/MyDrive/ncsrag_nat"
rng=random.Random(0); rows=[]
for f in sorted(glob.glob(f"{WORK}/judged_squad_*.json")):
    if "_partial" in f: continue
    rows+=json.load(open(f))
by={}
for r in rows: by.setdefault(r["cond"],[]).append(r)
per=max(1,120//max(1,len(by))); sample=[]
for cond,rs in by.items(): sample+=rng.sample(rs,min(per,len(rs)))
rng.shuffle(sample)
with open(f"{WORK}/human_annotation_template.csv","w",newline="") as fh:
    w=csv.writer(fh); w.writerow(["qid","cond","question","gold","answer","label_judge","human_label_correct_hallucination_abstain"])
    for r in sample: w.writerow([r["qid"],r["cond"],r["question"],r["gold"],r["answer"],r["label"],""])
print("wrote human_annotation_template.csv with",len(sample),"rows (fill the last column, then we compute kappa(judge,human))")


## After the run
Download from `MyDrive/ncsrag_nat/`: every `judged_squad_*.json`, `judge_summary.json`,
and `human_annotation_template.csv`. Put the `judged_*.json` files in `pilot_v2/data/`
and send `judge_summary.json`. Fill the blank column in the CSV (hand-label ~120 rows)
and send it back so we can report kappa(judge, human) and gate the headline labels at 0.7.